In [1]:
from pathlib import Path
import json
import re
import uuid
import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

c:\PokeAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1120 00:21:55.367000 14456 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


# Testes do RAG/Qdrant

## Teste 1: Montagem e Recuperação Simples.

### Gerando o banco

In [ ]:
from pathlib import Path
import json
import re
import uuid
import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

# Configurações
PASTA_DOCUMENTOS = Path("C:/PokeAI/databases/RAG/Metagame/documents")
PASTA_QDRANT = "C:/PokeAI/databases/RAG/Metagame/qdrant_db"
COLECAO = "pokeai_minilm_text"
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DIMENSAO = 384
LIMITE_TOKENS = 128
TAMANHO_LOTE = 64
K_RESULTADOS = 5

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo utilizado: {device}")

modelo = SentenceTransformer(MODELO, device=device)
modelo.max_seq_length = LIMITE_TOKENS

client = QdrantClient(path=PASTA_QDRANT)

# Funções para carregar documentos
def carregar_markdown(caminho):
    texto = caminho.read_text(encoding="utf-8-sig")
    linhas = texto.splitlines()
    registros = []
    titulo_atual = None
    conteudo_atual = []
    for linha in linhas:
        if re.match(r"^#\s+", linha):
            if conteudo_atual:
                conteudo = "\n".join(conteudo_atual).strip()
                if conteudo:
                    registros.append({"titulo": titulo_atual or caminho.stem, "texto": conteudo})
            titulo_atual = linha.lstrip("#").strip()
            conteudo_atual = [linha]
        else:
            conteudo_atual.append(linha)
    if conteudo_atual:
        conteudo = "\n".join(conteudo_atual).strip()
        if conteudo:
            registros.append({"titulo": titulo_atual or caminho.stem, "texto": conteudo})
    return registros

def carregar_jsonl(caminho):
    registros = []
    with caminho.open("r", encoding="utf-8-sig") as arquivo:
        for numero_linha, linha in enumerate(arquivo, start=1):
            linha = linha.strip()
            if not linha:
                continue
            try:
                dado = json.loads(linha)
                texto = json.dumps(dado, ensure_ascii=False, indent=2) if isinstance(dado, (dict, list)) else str(dado)
                titulo = dado.get("title", dado.get("titulo", dado.get("pokemon", f"Registro {numero_linha}"))) if isinstance(dado, dict) else f"Registro {numero_linha}"
            except json.JSONDecodeError:
                texto = linha
                titulo = f"Registro {numero_linha}"
            registros.append({"titulo": str(titulo), "texto": texto})
    return registros

def carregar_documentos():
    documentos = []
    arquivos = sorted(list(PASTA_DOCUMENTOS.rglob("*.md")) + list(PASTA_DOCUMENTOS.rglob("*.jsonl")) + list(PASTA_DOCUMENTOS.rglob("*.json")))
    for caminho in arquivos:
        if caminho.name.startswith("90"):
            print(f"Ignorando: {caminho.name}")
            continue
        if caminho.suffix.lower() == ".md":
            registros = carregar_markdown(caminho)
            tipo = "markdown"
        elif caminho.suffix.lower() in [".jsonl", ".json"]:
            registros = carregar_jsonl(caminho)
            tipo = caminho.suffix.lower().replace(".", "")
        else:
            continue
        for indice, registro in enumerate(registros):
            documentos.append({"id": str(uuid.uuid5(uuid.NAMESPACE_URL, f"{caminho.resolve()}_{indice}")), "arquivo": caminho.name, "caminho": str(caminho.relative_to(PASTA_DOCUMENTOS)), "registro": indice, "tipo": tipo, "titulo": registro["titulo"], "texto": registro["texto"]})
    return documentos

# Função para executar a indexação no Qdrant
def executar_indexacao():
    documentos = carregar_documentos()
    print(f"\nTotal de chunks encontrados: {len(documentos)}")
    if not documentos:
        print("Nenhum documento encontrado.")
        return
    if client.collection_exists(COLECAO):
        client.delete_collection(COLECAO)
        print(f"Coleção anterior '{COLECAO}' removida.")
    client.create_collection(collection_name=COLECAO, vectors_config=models.VectorParams(size=DIMENSAO, distance=models.Distance.COSINE))
    print(f"Coleção '{COLECAO}' criada.")
    for inicio in range(0, len(documentos), TAMANHO_LOTE):
        lote = documentos[inicio:inicio + TAMANHO_LOTE]
        textos = [documento["texto"] for documento in lote]
        embeddings = modelo.encode(textos, normalize_embeddings=True, show_progress_bar=False)
        pontos = []
        for documento, embedding in zip(lote, embeddings):
            payload = {"arquivo": documento["arquivo"], "caminho": documento["caminho"], "registro": documento["registro"], "tipo": documento["tipo"], "titulo": documento["titulo"], "texto": documento["texto"]}
            pontos.append(models.PointStruct(id=documento["id"], vector=embedding.tolist(), payload=payload))
        client.upsert(collection_name=COLECAO, points=pontos)
        print(f"Indexados {min(inicio + TAMANHO_LOTE, len(documentos))}/{len(documentos)} chunks.")
    print("\nIndexação concluída.")
    print(f"Total de pontos na coleção: {client.count(collection_name=COLECAO).count}")

# Executa tudo de fato
executar_indexacao()
client.close()

Dispositivo utilizado: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 561.72it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ignorando: 90_documents_description.json

Total de chunks encontrados: 4930
Coleção 'pokeai_minilm_text' criada.
Indexados 64/4930 chunks.
Indexados 128/4930 chunks.
Indexados 192/4930 chunks.
Indexados 256/4930 chunks.
Indexados 320/4930 chunks.
Indexados 384/4930 chunks.
Indexados 448/4930 chunks.
Indexados 512/4930 chunks.
Indexados 576/4930 chunks.
Indexados 640/4930 chunks.
Indexados 704/4930 chunks.
Indexados 768/4930 chunks.
Indexados 832/4930 chunks.
Indexados 896/4930 chunks.
Indexados 960/4930 chunks.
Indexados 1024/4930 chunks.
Indexados 1088/4930 chunks.
Indexados 1152/4930 chunks.
Indexados 1216/4930 chunks.
Indexados 1280/4930 chunks.
Indexados 1344/4930 chunks.
Indexados 1408/4930 chunks.
Indexados 1472/4930 chunks.
Indexados 1536/4930 chunks.
Indexados 1600/4930 chunks.
Indexados 1664/4930 chunks.
Indexados 1728/4930 chunks.
Indexados 1792/4930 chunks.
Indexados 1856/4930 chunks.
Indexados 1920/4930 chunks.
Indexados 1984/4930 chunks.
Indexados 2048/4930 chunks.
Indexad

### Puxando contexto

In [9]:
# Configurações
PASTA_QDRANT = "C:/PokeAI/databases/RAG/Metagame/qdrant_db"
COLECAO = "pokeai_minilm_text"
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Quantidade de trechos para recuperar
K  = 5

modelo = SentenceTransformer(MODELO, device="cuda")
client = QdrantClient(path=PASTA_QDRANT)

# Função de busca no Qdrant
def recuperar_contexto(pergunta, k=5):
    embedding = modelo.encode(pergunta, normalize_embeddings=True).tolist()
    resultados = client.query_points(collection_name=COLECAO, query=embedding, limit=k, with_payload=True)
    resultados = resultados.points
    if not resultados:
        print("Nenhum resultado encontrado.")
        return []
    print("=" * 90)
    print(f"PERGUNTA: {pergunta}")
    print("=" * 90)
    contextos = []
    for indice, resultado in enumerate(resultados, start=1):
        payload = resultado.payload
        print(f"\n--- Resultado {indice} ---")
        print(f"Arquivo: {payload.get('arquivo')}")
        print(f"Título: {payload.get('titulo')}")
        print(f"Tipo: {payload.get('tipo')}")
        print(f"Score vetorial: {resultado.score:.5f}")
        print("\nContexto recuperado:")
        print(payload.get("texto", ""))

        contextos.append({
            "arquivo": payload.get("arquivo"),
            "texto": payload.get("texto", "")
        })
    client.close()
    return contextos

# Altere a pergunta para testar a recuperação.
pergunta_teste = "Como o pokemon Sebleye usa a habilidade Prankster?"
contextos = recuperar_contexto(pergunta_teste, k=K)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1012.24it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PERGUNTA: Como o pokemon Sebleye usa a habilidade Prankster?

--- Resultado 1 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 307
Tipo: jsonl
Score vetorial: 0.64878

Contexto recuperado:
{
  "text": "Pokémon Ability: Poison Puppeteer. Effect: Pokémon poisoned by Pecharunt's moves will also become confused."
}

--- Resultado 2 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 170
Tipo: jsonl
Score vetorial: 0.60023

Contexto recuperado:
{
  "text": "Pokémon Ability: Magician. Effect: Steals the target's held item when the bearer uses a damaging move."
}

--- Resultado 3 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 158
Tipo: jsonl
Score vetorial: 0.58558

Contexto recuperado:
{
  "text": "Pokémon Ability: Prankster. Effect: Raises non-damaging moves' priority by one stage."
}

--- Resultado 4 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 182
Tipo: jsonl
Score vetorial: 0.56892

Contexto recuperado:
{
  "text": "Pokémon Ability: Pixilate. Effect: Tu

In [6]:
client.close()

In [16]:
contextos[0]

{'arquivo': '31_pokemon_abilities.jsonl',
 'texto': '{\n  "text": "Pokémon Ability: Poison Puppeteer. Effect: Pokémon poisoned by Pecharunt\'s moves will also become confused."\n}'}

In [ ]:
contextos[0]['arquivo']


'{\n  "text": "Pokémon Ability: Poison Puppeteer. Effect: Pokémon poisoned by Pecharunt\'s moves will also become confused."\n}'

In [30]:
contextos[5]['texto'].split('"text": "')[1].split('"\n}')[0]

IndexError: list index out of range

## Teste 2: Implementação com o arquivo 90 (descrição)

### Gerando o banco

In [ ]:
# Configurações
PASTA_DOCUMENTOS = Path("C:/PokeAI/databases/RAG/Metagame/documents")
PASTA_QDRANT = "C:/PokeAI/databases/RAG/Metagame/qdrant_db_90"
COLECAO = "pokeai_minilm_text_90"
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DIMENSAO = 384
LIMITE_TOKENS = 128
TAMANHO_LOTE = 64
K_RESULTADOS = 5

# ============================================================
# INICIALIZAÇÃO
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo utilizado: {device}")

modelo = SentenceTransformer(MODELO, device=device)
modelo.max_seq_length = LIMITE_TOKENS

client = QdrantClient(path=PASTA_QDRANT)

# ============================================================
# CARREGAMENTO DAS DESCRIÇÕES DO ARQUIVO 90
# ============================================================

def carregar_descricoes():
    arquivos_90 = list(PASTA_DOCUMENTOS.glob("90*.json"))
    if not arquivos_90:
        raise FileNotFoundError("Arquivo 90 de descrições não encontrado.")
    if len(arquivos_90) > 1:
        raise ValueError(f"Mais de um arquivo 90 encontrado: {arquivos_90}")
    with arquivos_90[0].open("r", encoding="utf-8-sig") as arquivo:
        descricoes = json.load(arquivo)
    print(f"Descrições carregadas: {len(descricoes)}")
    return descricoes

descricoes_arquivos = carregar_descricoes()

# ============================================================
# FUNÇÕES PARA CARREGAR DOCUMENTOS
# ============================================================

def carregar_markdown(caminho):
    texto = caminho.read_text(encoding="utf-8-sig")
    linhas = texto.splitlines()
    registros = []
    titulo_atual = None
    conteudo_atual = []
    for linha in linhas:
        if re.match(r"^#\s+", linha):
            if conteudo_atual:
                conteudo = "\n".join(conteudo_atual).strip()
                if conteudo:
                    registros.append({"titulo": titulo_atual or caminho.stem, "texto": conteudo})
            titulo_atual = linha.lstrip("#").strip()
            conteudo_atual = [linha]
        else:
            conteudo_atual.append(linha)
    if conteudo_atual:
        conteudo = "\n".join(conteudo_atual).strip()
        if conteudo:
            registros.append({"titulo": titulo_atual or caminho.stem, "texto": conteudo})
    return registros

def carregar_jsonl(caminho):
    registros = []
    with caminho.open("r", encoding="utf-8-sig") as arquivo:
        for numero_linha, linha in enumerate(arquivo, start=1):
            linha = linha.strip()
            if not linha:
                continue
            try:
                dado = json.loads(linha)
                texto = json.dumps(dado, ensure_ascii=False, indent=2) if isinstance(dado, (dict, list)) else str(dado)
                titulo = dado.get("title", dado.get("titulo", dado.get("pokemon", f"Registro {numero_linha}"))) if isinstance(dado, dict) else f"Registro {numero_linha}"
            except json.JSONDecodeError:
                texto = linha
                titulo = f"Registro {numero_linha}"
            registros.append({"titulo": str(titulo), "texto": texto})
    return registros

# ============================================================
# CARREGAMENTO E PREPARAÇÃO DOS DOCUMENTOS
# ============================================================

def carregar_documentos():
    documentos = []
    arquivos = sorted(list(PASTA_DOCUMENTOS.rglob("*.md")) + list(PASTA_DOCUMENTOS.rglob("*.jsonl")) + list(PASTA_DOCUMENTOS.rglob("*.json")))
    for caminho in arquivos:
        if caminho.name.startswith("90"):
            print(f"Ignorando arquivo de descrições: {caminho.name}")
            continue
        if caminho.suffix.lower() == ".md":
            registros = carregar_markdown(caminho)
            tipo = "markdown"
        elif caminho.suffix.lower() in [".jsonl", ".json"]:
            registros = carregar_jsonl(caminho)
            tipo = caminho.suffix.lower().replace(".", "")
        else:
            continue
        descricao = descricoes_arquivos.get(caminho.name, "")
        for indice, registro in enumerate(registros):
            texto_original = registro["texto"]
            texto_embedding = f"{descricao}\n\n{texto_original}".strip() if descricao else texto_original
            documentos.append({"id": str(uuid.uuid5(uuid.NAMESPACE_URL, f"{caminho.resolve()}_{indice}")), "arquivo": caminho.name, "caminho": str(caminho.relative_to(PASTA_DOCUMENTOS)), "registro": indice, "tipo": tipo, "titulo": registro["titulo"], "descricao_arquivo": descricao, "texto": texto_original, "texto_embedding": texto_embedding})
    return documentos

# ============================================================
# INDEXAÇÃO NO QDRANT
# ============================================================

def executar_indexacao():
    documentos = carregar_documentos()
    print(f"\nTotal de chunks encontrados: {len(documentos)}")
    print(f"Chunks com descrição adicional: {sum(1 for d in documentos if d['descricao_arquivo'])}")
    if not documentos:
        print("Nenhum documento encontrado.")
        return
    if client.collection_exists(COLECAO):
        client.delete_collection(COLECAO)
        print(f"Coleção anterior '{COLECAO}' removida.")
    client.create_collection(collection_name=COLECAO, vectors_config=models.VectorParams(size=DIMENSAO, distance=models.Distance.COSINE))
    print(f"Coleção '{COLECAO}' criada.")
    for inicio in range(0, len(documentos), TAMANHO_LOTE):
        lote = documentos[inicio:inicio + TAMANHO_LOTE]
        textos = [documento["texto_embedding"] for documento in lote]
        embeddings = modelo.encode(textos, normalize_embeddings=True, show_progress_bar=False)
        pontos = []
        for documento, embedding in zip(lote, embeddings):
            payload = {"arquivo": documento["arquivo"], "caminho": documento["caminho"], "registro": documento["registro"], "tipo": documento["tipo"], "titulo": documento["titulo"], "descricao_arquivo": documento["descricao_arquivo"], "texto": documento["texto"]}
            pontos.append(models.PointStruct(id=documento["id"], vector=embedding.tolist(), payload=payload))
        client.upsert(collection_name=COLECAO, points=pontos)
        print(f"Indexados {min(inicio + TAMANHO_LOTE, len(documentos))}/{len(documentos)} chunks.")
    print("\nIndexação concluída.")
    print(f"Total de pontos na coleção: {client.count(collection_name=COLECAO).count}")

# ============================================================
# BUSCA VETORIAL
# ============================================================

def buscar(pergunta, k=K_RESULTADOS):
    embedding = modelo.encode(pergunta, normalize_embeddings=True).tolist()
    resultados = client.query_points(collection_name=COLECAO, query=embedding, limit=k, with_payload=True)
    return resultados.points

def recuperar_contexto(pergunta, k=K_RESULTADOS):
    resultados = buscar(pergunta, k)
    if not resultados:
        print("Nenhum resultado encontrado.")
        return []
    print("=" * 90)
    print(f"PERGUNTA: {pergunta}")
    print("=" * 90)
    contextos = []
    for indice, resultado in enumerate(resultados, start=1):
        payload = resultado.payload
        print(f"\n--- Resultado {indice} ---")
        print(f"Arquivo: {payload.get('arquivo')}")
        print(f"Título: {payload.get('titulo')}")
        print(f"Tipo: {payload.get('tipo')}")
        print(f"Descrição: {payload.get('descricao_arquivo') or 'Não possui descrição'}")
        print(f"Score vetorial: {resultado.score:.5f}")
        print("\nContexto recuperado:")
        print(payload.get("texto", ""))
        contextos.append({"arquivo": payload.get("arquivo"), "titulo": payload.get("titulo"), "descricao": payload.get("descricao_arquivo"), "texto": payload.get("texto")})
    return contextos

# ============================================================
# EXECUÇÃO
# ============================================================

executar_indexacao()

# Pergunta de teste
pergunta_teste = "Quais são os teamates mais utilizados com o Kingambit?"
contextos = recuperar_contexto(pergunta_teste, k=K_RESULTADOS)
client.close()

Dispositivo utilizado: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2813.19it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Descrições carregadas: 19
Ignorando arquivo de descrições: 90_documents_description.json

Total de chunks encontrados: 4930
Chunks com descrição adicional: 1812
Coleção 'pokeai_minilm_text_90' criada.
Indexados 64/4930 chunks.
Indexados 128/4930 chunks.
Indexados 192/4930 chunks.
Indexados 256/4930 chunks.
Indexados 320/4930 chunks.
Indexados 384/4930 chunks.
Indexados 448/4930 chunks.
Indexados 512/4930 chunks.
Indexados 576/4930 chunks.
Indexados 640/4930 chunks.
Indexados 704/4930 chunks.
Indexados 768/4930 chunks.
Indexados 832/4930 chunks.
Indexados 896/4930 chunks.
Indexados 960/4930 chunks.
Indexados 1024/4930 chunks.
Indexados 1088/4930 chunks.
Indexados 1152/4930 chunks.
Indexados 1216/4930 chunks.
Indexados 1280/4930 chunks.
Indexados 1344/4930 chunks.
Indexados 1408/4930 chunks.
Indexados 1472/4930 chunks.
Indexados 1536/4930 chunks.
Indexados 1600/4930 chunks.
Indexados 1664/4930 chunks.
Indexados 1728/4930 chunks.
Indexados 1792/4930 chunks.
Indexados 1856/4930 chunks.
Ind

### Puxando contexto

In [18]:
# Configurações
PASTA_QDRANT = "C:/PokeAI/databases/RAG/Metagame/qdrant_db_90"
COLECAO = "pokeai_minilm_text_90"
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Quantidade de trechos para recuperar
K  = 5

modelo = SentenceTransformer(MODELO, device="cuda")
client = QdrantClient(path=PASTA_QDRANT)

# Função de busca no Qdrant
def recuperar_contexto(pergunta, k=5):
    embedding = modelo.encode(pergunta, normalize_embeddings=True).tolist()
    resultados = client.query_points(collection_name=COLECAO, query=embedding, limit=k, with_payload=True)
    resultados = resultados.points
    if not resultados:
        print("Nenhum resultado encontrado.")
        return []
    print("=" * 90)
    print(f"PERGUNTA: {pergunta}")
    print("=" * 90)
    contextos = []
    for indice, resultado in enumerate(resultados, start=1):
        payload = resultado.payload
        print(f"\n--- Resultado {indice} ---")
        print(f"Arquivo: {payload.get('arquivo')}")
        print(f"Título: {payload.get('titulo')}")
        print(f"Tipo: {payload.get('tipo')}")
        print(f"Score vetorial: {resultado.score:.5f}")
        print(f"Descrição: {payload.get('descricao_arquivo') or 'Não possui descrição'}")
        print("\nContexto recuperado:")
        print(payload.get("texto", ""))
        contextos.append(payload.get("texto", ""))
    client.close()
    return contextos

# Altere a pergunta para testar a recuperação.
pergunta_teste = "Como o pokemon Sebleye usa a habilidade Prankster?"
contextos = recuperar_contexto(pergunta_teste, k=K)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4220.16it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PERGUNTA: Como o pokemon Sebleye usa a habilidade Prankster?

--- Resultado 1 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 307
Tipo: jsonl
Score vetorial: 0.64878
Descrição: Não possui descrição

Contexto recuperado:
{
  "text": "Pokémon Ability: Poison Puppeteer. Effect: Pokémon poisoned by Pecharunt's moves will also become confused."
}

--- Resultado 2 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 170
Tipo: jsonl
Score vetorial: 0.60023
Descrição: Não possui descrição

Contexto recuperado:
{
  "text": "Pokémon Ability: Magician. Effect: Steals the target's held item when the bearer uses a damaging move."
}

--- Resultado 3 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 158
Tipo: jsonl
Score vetorial: 0.58558
Descrição: Não possui descrição

Contexto recuperado:
{
  "text": "Pokémon Ability: Prankster. Effect: Raises non-damaging moves' priority by one stage."
}

--- Resultado 4 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 182
Tipo: jsonl


## Testes de outros tipos de busca

In [ ]:
# Teste 2: Continuam recuperando os mesmos trechos
def recuperar_contexto(pergunta, k=5, variacoes=None, k_busca=10, constante=60):
    consultas = [pergunta] + (variacoes or [])
    resultados_combinados = {}

    for consulta in consultas:
        embedding = modelo.encode(consulta, normalize_embeddings=True).tolist()
        resultados = client.query_points(collection_name=COLECAO, query=embedding, limit=k_busca, with_payload=True).points

        for posicao, resultado in enumerate(resultados, start=1):
            ponto_id = resultado.id

            if ponto_id not in resultados_combinados:
                resultados_combinados[ponto_id] = {"resultado": resultado, "score_rrf": 0.0}

            resultados_combinados[ponto_id]["score_rrf"] += 1 / (constante + posicao)

    resultados_ordenados = sorted(resultados_combinados.values(), key=lambda x: x["score_rrf"], reverse=True)[:k]

    if not resultados_ordenados:
        print("Nenhum resultado encontrado.")
        return []

    print("=" * 90)
    print(f"PERGUNTA: {pergunta}")
    print("=" * 90)

    contextos = []

    for indice, item in enumerate(resultados_ordenados, start=1):
        resultado = item["resultado"]
        payload = resultado.payload

        print(f"\n--- Resultado {indice} ---")
        print(f"Arquivo: {payload.get('arquivo')}")
        print(f"Título: {payload.get('titulo')}")
        print(f"Tipo: {payload.get('tipo')}")
        print(f"Score RRF: {item['score_rrf']:.5f}")
        print("\nContexto recuperado:")
        print(payload.get("texto", ""))

        contextos.append(payload.get("texto", ""))

    return contextos

In [ ]:
# Teste 3: Puxou um outro negócio completamente aleatório
def recuperar_contexto(pergunta, k=5, k_busca=20, lambda_mmr=0.7):
    embedding = modelo.encode(pergunta, normalize_embeddings=True)
    resultados = client.query_points(collection_name=COLECAO, query=embedding.tolist(), limit=k_busca, with_payload=True).points

    if not resultados:
        print("Nenhum resultado encontrado.")
        return []

    textos = [r.payload.get("texto", "") for r in resultados]
    embeddings_docs = modelo.encode(textos, normalize_embeddings=True)

    relevancias = embeddings_docs @ embedding
    similaridades = embeddings_docs @ embeddings_docs.T

    selecionados = []
    candidatos = list(range(len(resultados)))

    while candidatos and len(selecionados) < k:
        melhor_indice = None
        melhor_score = float("-inf")

        for indice in candidatos:
            diversidade = max([similaridades[indice][s] for s in selecionados], default=0)
            score = lambda_mmr * relevancias[indice] - (1 - lambda_mmr) * diversidade

            if score > melhor_score:
                melhor_score = score
                melhor_indice = indice

        selecionados.append(melhor_indice)
        candidatos.remove(melhor_indice)

    print("=" * 90)
    print(f"PERGUNTA: {pergunta}")
    print("=" * 90)

    contextos = []

    for posicao, indice in enumerate(selecionados, start=1):
        resultado = resultados[indice]
        payload = resultado.payload

        print(f"\n--- Resultado {posicao} ---")
        print(f"Arquivo: {payload.get('arquivo')}")
        print(f"Título: {payload.get('titulo')}")
        print(f"Tipo: {payload.get('tipo')}")
        print(f"Score vetorial original: {resultado.score:.5f}")
        print(f"Relevância: {relevancias[indice]:.5f}")
        print("\nContexto recuperado:")
        print(textos[indice])

        contextos.append(textos[indice])

    return contextos


In [24]:
# Teste 4: 

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def recuperar_contexto(pergunta, k=5, k_busca=20, constante=60):
    embedding = modelo.encode(pergunta, normalize_embeddings=True).tolist()

    resultados = client.query_points(collection_name=COLECAO, query=embedding, limit=k_busca, with_payload=True).points

    if not resultados:
        print("Nenhum resultado encontrado.")
        return []

    textos = [r.payload.get("texto", "") for r in resultados]

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    matriz = vectorizer.fit_transform(textos + [pergunta])
    scores_lexicais = (matriz[:-1] @ matriz[-1].T).toarray().flatten()

    ranking_lexical = np.argsort(scores_lexicais)[::-1]
    ranking_vetorial = list(range(len(resultados)))

    pontuacoes = {}

    for posicao, indice in enumerate(ranking_vetorial, start=1):
        pontuacoes[indice] = pontuacoes.get(indice, 0) + 1 / (constante + posicao)

    for posicao, indice in enumerate(ranking_lexical, start=1):
        pontuacoes[indice] = pontuacoes.get(indice, 0) + 1 / (constante + posicao)

    melhores = sorted(pontuacoes, key=pontuacoes.get, reverse=True)[:k]

    print("=" * 90)
    print(f"PERGUNTA: {pergunta}")
    print("=" * 90)

    contextos = []

    for posicao, indice in enumerate(melhores, start=1):
        resultado = resultados[indice]
        payload = resultado.payload

        print(f"\n--- Resultado {posicao} ---")
        print(f"Arquivo: {payload.get('arquivo')}")
        print(f"Título: {payload.get('titulo')}")
        print(f"Tipo: {payload.get('tipo')}")
        print(f"Score vetorial: {resultado.score:.5f}")
        print(f"Score lexical: {scores_lexicais[indice]:.5f}")
        print(f"Score combinado: {pontuacoes[indice]:.5f}")
        print("\nContexto recuperado:")
        print(textos[indice])

        contextos.append(textos[indice])

    return contextos


In [25]:
# Configurações
PASTA_QDRANT = "C:/PokeAI/databases/RAG/Metagame/qdrant_db"
COLECAO = "pokeai_minilm_text"
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Quantidade de trechos para recuperar
K  = 5
pergunta_teste = "Como o pokemon Sebleye usa a habilidade Prankster?"

modelo = SentenceTransformer(MODELO, device="cuda")
client = QdrantClient(path=PASTA_QDRANT)

print('-'*50, 'Avaliação do NORMAL', '-'*50)
contextos = recuperar_contexto(pergunta_teste, k=K)
client.close()
PASTA_QDRANT = "C:/PokeAI/databases/RAG/Metagame/qdrant_db_90"
COLECAO = "pokeai_minilm_text_90"
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

modelo = SentenceTransformer(MODELO, device="cuda")
client = QdrantClient(path=PASTA_QDRANT)
print('-'*50, 'Avaliação do COM DESCIÇÃO', '-'*50)
contextos = recuperar_contexto(pergunta_teste, k=K)
client.close()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 750.20it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-------------------------------------------------- Avaliação do NORMAL --------------------------------------------------
PERGUNTA: Como o pokemon Sebleye usa a habilidade Prankster?

--- Resultado 1 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 158
Tipo: jsonl
Score vetorial: 0.58558
Score lexical: 0.05927
Score combinado: 0.03227

Contexto recuperado:
{
  "text": "Pokémon Ability: Prankster. Effect: Raises non-damaging moves' priority by one stage."
}

--- Resultado 2 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 307
Tipo: jsonl
Score vetorial: 0.64878
Score lexical: 0.00000
Score combinado: 0.02889

Contexto recuperado:
{
  "text": "Pokémon Ability: Poison Puppeteer. Effect: Pokémon poisoned by Pecharunt's moves will also become confused."
}

--- Resultado 3 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 170
Tipo: jsonl
Score vetorial: 0.60023
Score lexical: 0.00000
Score combinado: 0.02879

Contexto recuperado:
{
  "text": "Pokémon Ability: Magician.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2676.88it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-------------------------------------------------- Avaliação do COM DESCIÇÃO --------------------------------------------------
PERGUNTA: Como o pokemon Sebleye usa a habilidade Prankster?

--- Resultado 1 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 158
Tipo: jsonl
Score vetorial: 0.58558
Score lexical: 0.05927
Score combinado: 0.03227

Contexto recuperado:
{
  "text": "Pokémon Ability: Prankster. Effect: Raises non-damaging moves' priority by one stage."
}

--- Resultado 2 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 307
Tipo: jsonl
Score vetorial: 0.64878
Score lexical: 0.00000
Score combinado: 0.02889

Contexto recuperado:
{
  "text": "Pokémon Ability: Poison Puppeteer. Effect: Pokémon poisoned by Pecharunt's moves will also become confused."
}

--- Resultado 3 ---
Arquivo: 31_pokemon_abilities.jsonl
Título: Registro 170
Tipo: jsonl
Score vetorial: 0.60023
Score lexical: 0.00000
Score combinado: 0.02879

Contexto recuperado:
{
  "text": "Pokémon Ability: Mag